# AF MEGADADOS 25-2
**NOME**: SEU NOME AQUI!

## Permissões de uso

Proibido publicar ou compartilhar este material com terceiros.

## Regras
- Esta prova é formada por 5 (cinco) questões e tem **duração de 2h00**.
- Os exercícios com correção automática não terão correção manual, mas poderá haver checks de qualidade/honestidade. Caso tente burlar os testes, sua nota de prova será zero e será enviado ao regime disciplinar. O que é burlar? Por exemplo, para a pergunta "calcule a média da coluna x", o correto é fazer "SELECT AVG(x) FROM tabela", já fazer "SELECT 100.10" é considerado burlar os testes, pois sua query não funciona de forma genérica, apenas cola o próprio valor de resultado.
- Você pode trazer uma folha de papel (tamanho próximo a A4 / oficio) preenchida dos dois lados e utilizar como consulta. A folha pode ser tanto impressa quanto escrita a mão, mas precisa ser física (não pode acessar no computador). Não é permitido utilizar lupa rsrs.
- Será permitido utilizar o Workbench para testar queries e fazer forward / reverse engineering. Feche as abas com respostas realizadas durante o semestre.
- Não é permitido compartilhar qualquer material durante a prova.
- Não serão permitidas consultas a outros meios (material blackboard, sites, notebooks das aulas, github) impressos ou digitais, nem contactar outras pessoas. Caso ocorra, a nota da prova será zero e será enviado ao regime disciplinar.
- Não será permitido o uso de ferramentas de IA (copilot, chatgpt e afins). Você precisará providenciar um ambiente seguro para a prova, sem extensões instaladas (não basta desativar) nem sites abertos. Caso ocorra, a nota da prova será zero e será enviado ao regime disciplinar.

## Entrega

Não se esqueça de **entregar** o notebook ao final da prova. É obrigatório deixar as células **executadas**!

## Insper autograding!

Para receber feedback dos exercícios, iremos utilizar o `insper autograding`.

**Importante**: você precisará do `.env`!

In [1]:
!pip install -U git+https://github.com/macielcalebe/insperautograding.git

  Cloning https://github.com/macielcalebe/insperautograding.git to /tmp/pip-req-build-3db83l4d
  Running command git clone --filter=blob:none --quiet https://github.com/macielcalebe/insperautograding.git /tmp/pip-req-build-3db83l4d
  Resolved https://github.com/macielcalebe/insperautograding.git to commit 49f1176548d7560b38d0314e1115f93467141893
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/124.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.6 MB/s eta 0:00:00a 0:00:01
  Created wheel for insperautograder: filename=insperautograder-0.4.0-py3-none-any.whl size=4568 sha256=6b8ebf9be1552cfd6c59a14490e7a54c5d51a272d10682444ae2e4101d908ae8
  Stored in directory: /tmp/pip-ephem-wheel-cache-0qyx4mvk/wheels/bb/01/cc/7f397587bfedd6c633e2627496c45d5693aa959ee8237ead1f
Successfully built insperautograder
  Attempting uninstall: jupyterl

## Como resolver os exercícios?

No exercício 1, rode o Docker para resolver as questões de programação funcional.

Nas demais, crie a base da prova em sua máquina. Utilize o notebook, MySQL Workbench ou o conector para testar as queries e soluções. Quando estiver bastante certo de que a resposta está correta, faça a submissão para o servidor.
Para macOS e linux, utilize:

```bash
docker run \
    -it \
    --rm \
    -p 8888:8888 \
    -p 4040:4040 \
    -v "`pwd`":/home/jovyan/work \
    jupyter/pyspark-notebook
```

Se estiver no Windows estes comandos, utilize:

- No Powershell: `docker run -it --rm -p 8888:8888 -p 4040:4040 -v ${PWD}:/home/jovyan/work jupyter/pyspark-notebook`

- No Prompt de comando: `docker run -it --rm -p 8888:8888 -p 4040:4040 -v %cd%:/home/jovyan/work jupyter/pyspark-notebook`

Agora abra esse notebook lá no container!

## Import das bibliotecas

Vamos realizar o import das bibliotecas.

In [3]:
import os
import insperautograder.jupyter as ia
from functools import partial
from pyspark.sql import SparkSession
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [4]:
spark = SparkSession.builder.appName("AF_MD_25_2").getOrCreate()
sc = spark.sparkContext

### Notas

A prova vale 10,5 pontos:

- As primeiras três questões, que possuem correção automática, valem 7 pontos.
- A questão 4 vale 3 pontos.
- A questão 5 é de nota extra (0,5 pontos).

Na API de correção automática a nota de cada questão será ponderada pelo seu peso. A nota será apresentada no intervalo 0 a 10, multiplique por 0.70 para saber a nota final considerando toda a prova.

Para conferir a nota da correção automática da prova, utilize:

In [ ]:
ia.grades(task="af_md_25_2")

In [ ]:
ia.grades(by="TASK", task="af_md_25_2")

**Exercício 1 - Analytics de Podcasts**

<img src="img/podcast_1.jpeg">

A partir dos dados transacionais da plataforma Xtremo (descrita no exercício 2), foi criado um arquivo para análise dos podcasts.

O formato de cada linha do arquivo é:

```python
"<ID_PODCAST>,<NOME_PODCAST>,<STATUS>,<NOME_HOST_LIDER>,<NOME_HOST_2>,...,<NOME_HOST_N>"
```

Onde:

- `<ID_PODCAST>`: número inteiro que identifica o podcast.
- `<NOME_PODCAST>`: nome do podcast.
- `<STATUS>`: status do podcast. Por exemplo: Ativo, Inativo. Outros status podem existir.
- `<NOME_HOST_LIDER>`: nome do host líder do podcast.
- `<NOME_HOST_2>`: nome do segundo host do podcast (se houver).
- ...
- `<NOME_HOST_N>`: nome do enésimo host do podcast (se houver).

**Dica:** repare bem o padrão de separadores seguido por todas as linhas (vírgula `,`)!

Você encontrará exemplos nos arquivos `data/podcasts_*.csv`. Utilize estes arquivos como base para os exercícios.

**a)** Crie uma função `podcast_com_mais_hosts` que recebe (nesta ordem):

- o RDD no formato acima.

E retorna uma tupla `(nome_podcast, k)` com o nome do podcast com maior número de membros (hosts), seguido da quantidade de hosts.

**Obs**: Não que faça diferença neste exercício, mas sempre que falarmos dos nomes dos hosts, considere que o host líder também conta!

In [17]:
def podcast_com_mais_hosts(rdd):
    mapeado = rdd.map(lambda x: x.split(',')).map(lambda x: (x[1],x[3:]))
    mapeado2 = mapeado.map(lambda x: (x[0],len(x[1])))
    reduzido = mapeado2.reduce(lambda a,b: a if a[1] > b[1] else b)
    return reduzido

rdd = sc.textFile("data/podcasts_1.csv")

# Descomente caso queira executar localmente
print(podcast_com_mais_hosts(rdd))

('All Pacas', 5)


Um teste local:

In [19]:
# Descomente caso queira executar localmente
# # Garantir uso do RDD correto
rdd = sc.textFile("data/podcasts_2.csv")
assert podcast_com_mais_hosts(rdd) == ('Mestres do Codigo', 8)

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [ ]:
ia.sender(answer="podcast_com_mais_hosts", task="af_md_25_2", question="ex01a", answer_type="pycode")

**b)** Crie uma função `freq_absoluta_status(rdd, status)` que recebe um RDD no formato descrito e uma string `status`.

Retorne quantos podcasts com o status fornecido existem (ou seja, a frequência absoluta).

In [21]:
def freq_absoluta_status(rdd, status):
    mapeado = rdd.map(lambda x: x.split(',')).map(lambda x: x[2]).filter(lambda x: x ==status).count()
    return mapeado

rdd = sc.textFile("data/podcasts_1.csv")

# Descomente caso queira executar localmente
print(freq_absoluta_status(rdd, "Ativo"))

0


Alguns testes locais:

In [22]:
# Descomente caso queira executar localmente

# Garantir uso do RDD 2
rdd2 = sc.textFile("data/podcasts_2.csv")

assert freq_absoluta_status(rdd2, "Ativo") == 7
assert freq_absoluta_status(rdd2, "Inativo") == 3

# Garantir uso do RDD 6
rdd6 = sc.textFile("data/podcasts_6.csv")

assert freq_absoluta_status(rdd6, "No ar") == 30

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

**c)** Crie uma função `freq_nomes(rdd, status, k)` que recebe o RDD no formato descrito, um `status` (string) e um inteiro `k`.

Considerando apenas os podcasts com o **status** definido, a função deve calcular a **frequência relativa** de cada **nome de host** e retornar uma lista de tuplas no formato `(nome_host, frequencia)`, ordenadas de forma decrescente pela frequência. O parâmetro `k` deve ser utilizado para limitar o número de resultados retornados.

**Obs**:

- Compare os nomes de forma *case-sensitive* (maiúsculas e minúsculas não são equivalentes).
- A frequência relativa deverá ser representada como um valor percentual inteiro entre **0 e 100**, arredondado para cima com `math.ceil`.
- O servidor de correção já tem a biblioteca `math` importada.

**Atenção**: Esta questão pode ser mais complexa. Recomendamos que você a resolva por último!

In [44]:
import math

def freq_nomes(rdd, status, k):
    mapeado = rdd.map(lambda x: x.split(',')).map(lambda x: (x[2],x[3:])).filter(lambda x: x[0]==status)
    apareceu = mapeado.map(lambda x: x[1])
    mapeado2 = apareceu.flatMap(lambda x: x)
    total = mapeado2.count()
    resposta = mapeado2.map(lambda x: (x,1)).reduceByKey(lambda a,b: a+b)
    resposta2 = resposta.map(lambda x: (x[0],(math.ceil(x[1]/total*100))))
    return resposta2.takeOrdered(k, lambda x: -x[1])

# Descomente caso queira executar localmente
rdd = sc.textFile("data/podcasts_2.csv")
print(freq_nomes(rdd, 'Ativo', 3))

[('Alice', 16), ('Igor', 12), ('Bob', 10)]


Alguns testes locais:

In [45]:
# Descomente caso queira executar localmente

rdd = sc.textFile("data/podcasts_2.csv")

result = freq_nomes(rdd, "Ativo", 3)
assert all(isinstance(x[0], str) for x in result)
assert all(isinstance(x[1], int) for x in result)
assert result == [('Alice', 16), ('Igor', 12), ('Bob', 10)]

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

Agora que você terminou a parte de programação funcional da prova, salve o notebook e continue no VS Code.

## Plataforma de Podcast XTREMO!

No próximo exercício, iremos trabalhar com a base de dados **XTREMO**.

O banco de dados `xtremo` é um sistema relacional sintético projetado para **gerenciar informações de um ecossistema de podcasts**. O foco é manter o registro de aspectos da produção, distribuição e monetização de conteúdo de áudio. Ele contém tabelas que cobrem as seguintes áreas principais:

- **Conteúdo:** A tabela **`podcast`** armazena os dados principais dos programas, enquanto a tabela **`episodio`** detalha cada episódio.
- **Participantes:** O banco gerencia informações sobre os **`hosts`** (apresentadores) e **`convidados`**, incluindo detalhes de contato e participação. A tabela **`podcast_host`** relaciona hosts a podcasts, e **`episodio_convidado`** liga convidados a episódios.
- **Monetização:** As tabelas **`anunciante`**, **`tipo_anuncio`**, e **`anuncio`** lidam com dados comerciais. A tabela **`anuncio_episodio`** associa anúncios a episódios específicos, registrando informações como valor pago e ouvintes no momento da veiculação.
- **Distribuição e Engajamento:** As tabelas **`plataforma_distribuicao`** e **`episodio_plataforma`** controlam a presença dos episódios em diferentes plataformas. A tabela **`topico`** e **`episodio_topico`** permitem categorizar o conteúdo e as **`mensagens`** coletam dados de interação do público.

### Instalação da base

Execute os scripts `xtremo_0001.sql` e `xtremo_0002.sql` no **MySQL Workbench**. Estes scripts criam um banco `xtremo` e inserem dados de exemplo para resolução da prova.

O banco de dados pode ser representado pelo seguinte diagrama do modelo relacional (também disponível em PDF na pasta `docs`):

<img src="img/xtremo.png">

Vamos criar nosso HELPER de conexão com o banco!

In [ ]:
import mysql.connector
from functools import partial
import os
import insperautograder.jupyter as ia
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

def get_connection_helper():

    def run_db_query(connection, query, args=None):
        with connection.cursor() as cursor:
            print("Executando query:")
            for result in cursor.execute(query, multi=True):
                if result.with_rows:
                    for row in result.fetchall():
                        print(row)
                else:
                    print(f"{result.rowcount} linhas afetadas.")

    connection = mysql.connector.connect(
        host=os.getenv("MD_DB_SERVER"),
        user=os.getenv("MD_DB_USERNAME"),
        password=os.getenv("MD_DB_PASSWORD"),
        database="xtremo",
    )
    return connection, partial(run_db_query, connection)


connection, db = get_connection_helper()

**Exercício 2 - Análise de Receita de Anúncios**:

A equipe de analytics da Xtremo precisa frequentemente calcular a receita total gerada por anúncios de um determinado anunciante em um período específico.

Crie uma função `receita_anunciante` no MySQL que recebe:
- `p_id_anunciante`: ID do anunciante (INT)
- `p_data_inicio`: data de início do período (DATE)
- `p_data_fim`: data de fim do período (DATE)

A função deve retornar o valor total pago (`valor_pago`) pelos anúncios deste anunciante que foram lidos completamente (`lido_completo = 1`) no período especificado (intervalo fechado).

Retorne `DECIMAL(12,2)`. Se não houver anúncios no período, retorne `0.00`.

**SQL de referência:**

```sql
DROP FUNCTION IF EXISTS calcula_algo;

CREATE FUNCTION calcula_algo(p_id_pessoa INT)
RETURNS DECIMAL(5,2)
READS SQL DATA
BEGIN
    DECLARE valor DECIMAL(5,2);

    RETURN 0.0;
END;
```

**Dica**:

- Considere a coluna `momento_leitura` da tabela `anuncio_episodio` para verificar se o anúncio está no período.
- O valor a ser somado é o `valor_pago` da tabela `anuncio_episodio`.
- Garanta que está retornando no formato (tipo de dados) esperado.
- Utilize `DATE(x.algum_campo_datetime)` para extrair a parte da data (sem hora) de um datetime.

In [ ]:
sql_ex02 = """
DROP FUNCTION IF EXISTS receita_anunciante;

CREATE FUNCTION receita_anunciante(p_id_anunciante INT, p_data_inicio DATE, p_data_fim DATE)
RETURNS DECIMAL(12,2)
READS SQL DATA
BEGIN
    DECLARE valor_pago DECIMAL(12,2);
    
    
    RETURN valor_pago;
END
"""

# Descomente caso queira executar localmente
# db(sql_ex02)

Exemplo de uso:

In [ ]:
# Descomente caso queira executar localmente
# db("SELECT receita_anunciante(1, '2023-06-01', '2023-08-01')")

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [ ]:
ia.sender(answer="sql_ex02", task="af_md_25_2", question="ex02", answer_type="pyvar")

**Exercício 3 - Estimativa de Armazenamento de Áudio**:

<img src="img/storage.jpeg">

A Xtremo precisa estimar quanto espaço de armazenamento será necessário para hospedar os episódios de podcasts em diferentes qualidades de áudio.

Áudios podem ser armazenados em diferentes *bitrates* (taxa de bits por segundo, quantos bits são necessários para representar um segundo de áudio), que afetam tanto a qualidade quanto o tamanho do arquivo.

Crie uma função `estimar_armazenamento_podcast(duracao_minutos, bitrate_kbps, num_episodios)` que calcula o espaço total necessário em GB para armazenar uma temporada de podcasts.

**Parâmetros:**
- `duracao_minutos`: duração média de cada episódio em minutos (int)
- `bitrate_kbps`: taxa de bits em kilobits por segundo (int)
- `num_episodios`: número total de episódios (int)


**Obs:**
- Retorne o valor em GB arredondado para cima usando `math.ceil(valor)`.
- Considere: 1 byte = 8 bits, 1 KB = 1000 bytes, 1 MB = 1000 KB, 1 GB = 1000 MB.

In [ ]:
import math
def estimar_armazenamento_podcast(duracao_minutos, bitrate_kbps, num_episodios):
    tamanho_total_gb = 0

    return math.ceil(tamanho_total_gb)

Exemplos de execução:

In [ ]:
# Descomente caso queira executar localmente

# print(estimar_armazenamento_podcast(60, 128, 100))  # 100 eps de 60min a 128kbps
# print(estimar_armazenamento_podcast(30, 256, 50))   # 50 eps de 30min a 256kbps

Alguns testes locais:

In [ ]:
# Descomente caso queira executar localmente

# assert estimar_armazenamento_podcast(60, 128, 100) == 6
# assert estimar_armazenamento_podcast(30, 256, 50) == 3

Após testar localmente e considerar sua solução correta, faça o envio clicando no botão abaixo!

In [ ]:
ia.sender(answer="estimar_armazenamento_podcast", task="af_md_25_2", question="ex03", answer_type="pycode")

**Exercício 4 - Sistema de Apoio a Criadores**: (**Nota: 3,0**)

<img src="img/send_money.jpg">

A Xtremo quer implementar um sistema de financiamento coletivo (estilo Patreon ou Apoia.se) onde usuários podem apoiar financeiramente seus canais favoritos em troca de recompensas.

Atualmente, os dados de apoios estão armazenados em uma única tabela denormalizada conforme mostrado abaixo:

| id_apoio | data_inicio_apoio | id_usuario | nome_usuario | email_usuario | id_canal | nome_canal | tags_canal | id_nivel_recompensa | nome_nivel | valor_mensal | descricao_nivel | beneficios |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 1 | 2024-01-10 | 501 | Lucas Mendes | lucas@email.com | 10 | Ciência Todo Dia | Ciência, Educação | 1 | Apoio Júnior | 15.00 | Acesso ao Discord | ['Badge Exclusivo', 'Chat Privado'] |
| 2 | 2024-02-15 | 502 | Beatriz Lima | bia@email.com | 10 | Ciência Todo Dia | Ciência, Educação | 2 | Apoio Pleno | 30.00 | Vídeos Antecipados | ['Badge Exclusivo', 'Chat Privado', 'Vídeo Extra'] |
| 3 | 2024-03-20 | 501 | Lucas Mendes | lucas@email.com | 20 | História Viva | História, Cultura | 5 | Mecenas | 50.00 | Nome nos Créditos | ['Livro Digital', 'Encontro Mensal'] |
| 4 | 2024-04-05 | 503 | Fernando O. | fer@email.com | 20 | História Viva | História, Cultura | 5 | Mecenas | 50.00 | Nome nos Créditos | ['Livro Digital', 'Encontro Mensal'] |
| 5 | 2024-04-10 | 502 | Beatriz Lima | bia@email.com | 30 | Tech News | Tecnologia | 8 | Insider | 20.00 | Newsletter Semanal | ['News VIP'] |

**Descrição dos Campos:**

- `id_apoio (PK)`: Identificador único do registro de apoio.
- `data_inicio_apoio`: Data em que o usuário começou a apoiar.
- `id_usuario`: Identificador do usuário apoiador. Um usuário pode apoiar diversos canais.
- `nome_usuario`: Nome completo do usuário. Considere que não é necessário consultar nome e sobrenome separadamente.
- `email_usuario`: Email do usuário.
- `id_canal`: Identificador do canal/criador sendo apoiado.
- `nome_canal`: Nome do canal sendo apoiado.
- `tags_canal`: Tags ou categorias do canal sendo apoiado (múltiplas tags separadas por vírgula). Considere que é necessário consultar cada tag individualmente.
- `id_nivel_recompensa`: Identificador do nível de apoio escolhido (tier). Cada apoio refere-se a um nível específico. Cada canal tem seus próprios níveis de apoio.
- `nome_nivel`: Nome do nível de apoio (ex: "Apoio Júnior", "Mecenas").
- `valor_mensal`: Valor monetário cobrado mensalmente neste nível. Todos que estejam em um mesmo nível pagam o mesmo valor.
- `descricao_nivel`: Descrição textual geral das recompensas do nível.
- `beneficios`: Lista de benefícios específicos incluídos neste nível de recompensa. Considere que é necessário consultar quais níveis oferecem um benefício específico. Considere que os benefícios são compartilhados entre os níveis.

**Tarefa**:

Normalize os dados até a 3ª Forma Normal (3NF), garantindo:

1. Eliminação de redundâncias.
1. Tratamento correto de atributos multivalorados.
1. Integridade referencial entre as entidades.
1. Relacionamentos corretos.

Crie no **MySQL Workbench** o **Diagrama do Modelo Relacional (Diagrama de Entidades e Relacionamentos Extendido [DEER])** que representa a base de dados normalizada.

**Instruções:**
- Salve o DER na pasta `resposta_ex4`. Utilize formato **png** ou **jpg**.
- Insira o caminho do arquivo no notebook.

**Critérios de Avaliação (Rubrica):**

| Conceito | Nota Porcentual | Descrição |
|:----------:|----------:|:---------|
| I | 0.0 | Insuficiente, não considera as informações a serem armazenadas ou cenário diferente do proposto |
| D | 0.3 | Não está na 1NF, mas solução em desenvolvimento |
| C | 0.6 | Está na 1NF, sem erros graves |
| B | 0.8 | Está na 2NF, sem erros |
| A | 1.0 | Está na 3NF, sem erros, redundâncias ou ineficiências |

<div class="alert alert-success">

Seu diagrama do workbench AQUI!

<img src="resposta_ex04/exemplo.png">

</div>

<div class="alert alert-success">

Caso considere necessário, deixe AQUI uma explicação sobre as decisões que tomou neste exercício!

</div>

**Exercício 5)** (**Nota EXTRA: 0,5**) Seu colega (um mero mortal) realiza a seguinte pergunta:

*"Por que utilizar um SGBD relacional se posso salvar os dados do meu sistema gerencial em um CSV?"*

Responda de forma crítica. Justifique detalhadamente, de forma aprofundada.

Esta questão será corrigida considerando a seguinte rubrica:

| Conceito | Percentual da Nota | Descrição                                                                                                                                               |
|:----------:|----------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------|
| I        | 0.0 |Apenas citou o assunto ou alguns fatos sem explicações                                                                                                  |
| D        | 0.3 |Explicou superficialmente o assunto ou fatos mas sem muitos detalhes conclusivos                                                                        |
| C        | 0.6 |Explicou com detalhes, apresentando definições concretas.                                                                                               |
| B        | 0.8 |Explicou com detalhes, apresentando definições concretas e exemplos de uso.                                                                            |
| A        | 1.0 |Explicou com detalhes, apresentando definições concretas, exemplos de uso e ainda outros tópicos correlatos, fazendo uma conexão lógica entre eles. |

## Referências das imagens

Filtradas para licença Creative Commons no Google:

- https://cdn.prod.website-files.com/637859b9a818d7ddb05bb5fb/64d06465a8410d89a3782469_637fc63ff88e999b5ddca8b0_Starting-a-Business-Podcast.jpeg
- https://www.sectorlink.com/img/blog/server-ai.jpg
- https://www.publicdomainpictures.net/pictures/270000/velka/money-transfer-mobile-banking-bu.jpg

### Conferindo as notas!

In [ ]:
ia.grades(task="af_md_25_2")

In [ ]:
ia.grades(by="TASK", task="af_md_25_2")

## Entrega!

É hora de entregar. Faça **zip** (não RAR), envie no Blackboard e finalize o teste no proctorio!

<div class="alert alert-success">

Caso considere necessário, deixe AQUI comentários ou reclamações sobre qualquer questão da prova!

</div>